# Denoising Autoencoder on MNIST (Pure NumPy)
Please ensure `mnist_train.csv` and `mnist_test.csv` (from the Kaggle dataset) are placed in the same directory as this notebook to load the data manually.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load dataset manually from CSV files
train_data = pd.read_csv('mnist_train.csv').values
test_data = pd.read_csv('mnist_test.csv').values

# Separate labels from pixels and normalize
X_train = train_data[:, 1:] / 255.0
X_test = test_data[:, 1:] / 255.0

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")


## 2. Add Artificial Noise

In [ ]:
noise_factor = 0.5
X_train_noisy = np.clip(X_train + noise_factor * np.random.randn(*X_train.shape), 0., 1.)
X_test_noisy = np.clip(X_test + noise_factor * np.random.randn(*X_test.shape), 0., 1.)

plt.figure(figsize=(10, 2))
for i in range(5):
    ax = plt.subplot(1, 5, i + 1)
    plt.imshow(X_train_noisy[i].reshape(28, 28), cmap='gray')
    ax.axis('off')
plt.show()


## 3. Model Architecture

In [ ]:
input_size, hidden_size, output_size = 784, 128, 784
learning_rate, epochs, batch_size = 0.005, 30, 128

W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2. / input_size)
b1 = np.zeros(hidden_size)
W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2. / hidden_size)
b2 = np.zeros(output_size)

def relu(x): return np.maximum(0, x)
def relu_deriv(x): return (x > 0).astype(float)
def sigmoid(x): return 1. / (1. + np.exp(-np.clip(x, -500, 500)))
def sigmoid_deriv(x): s = sigmoid(x); return s * (1 - s)

m_W1, v_W1 = np.zeros_like(W1), np.zeros_like(W1)
m_b1, v_b1 = np.zeros_like(b1), np.zeros_like(b1)
m_W2, v_W2 = np.zeros_like(W2), np.zeros_like(W2)
m_b2, v_b2 = np.zeros_like(b2), np.zeros_like(b2)
beta1, beta2, epsilon = 0.9, 0.999, 1e-8
t = 0


## 4. Training

In [ ]:
num_samples = X_train.shape[0]
loss_history = []

for epoch in range(epochs):
    indices = np.arange(num_samples)
    np.random.shuffle(indices)
    
    total_loss = 0
    for start_idx in range(0, num_samples, batch_size):
        batch_idx = indices[start_idx:start_idx + batch_size]
        X_batch_noisy = X_train_noisy[batch_idx]
        X_batch_clean = X_train[batch_idx]
        
        Z1 = X_batch_noisy.dot(W1) + b1
        A1 = relu(Z1)
        Z2 = A1.dot(W2) + b2
        A2 = sigmoid(Z2)
        
        loss = np.mean((A2 - X_batch_clean) ** 2)
        total_loss += loss * len(batch_idx)
        
        dA2 = 2.0 * (A2 - X_batch_clean) / len(batch_idx)
        dZ2 = dA2 * sigmoid_deriv(Z2)
        dW2 = A1.T.dot(dZ2)
        db2 = np.sum(dZ2, axis=0)
        
        dA1 = dZ2.dot(W2.T)
        dZ1 = dA1 * relu_deriv(Z1)
        dW1 = X_batch_noisy.T.dot(dZ1)
        db1 = np.sum(dZ1, axis=0)
        
        t += 1
        for param, dparam, m, v in zip(
            [W1, b1, W2, b2], [dW1, db1, dW2, db2],
            [m_W1, m_b1, m_W2, m_b2], [v_W1, v_b1, v_W2, v_b2]
        ):
            m[:] = beta1 * m + (1 - beta1) * dparam
            v[:] = beta2 * v + (1 - beta2) * (dparam ** 2)
            m_hat = m / (1 - beta1 ** t)
            v_hat = v / (1 - beta2 ** t)
            param -= learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)
            
    epoch_loss = total_loss / num_samples
    loss_history.append(epoch_loss)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{epochs}, Loss: {epoch_loss:.4f}")

plt.plot(loss_history)
plt.title("Training Loss")
plt.show()


## 5. Evaluation

In [ ]:
Z1_test = X_test_noisy.dot(W1) + b1
A1_test = relu(Z1_test)
Z2_test = A1_test.dot(W2) + b2
A2_test = sigmoid(Z2_test)

n = 10
plt.figure(figsize=(20, 6))
for i in range(n):
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')
    ax.axis('off')
    
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(X_test_noisy[i].reshape(28, 28), cmap='gray')
    ax.axis('off')
    
    ax = plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(A2_test[i].reshape(28, 28), cmap='gray')
    ax.axis('off')

plt.tight_layout()
plt.show()
